In [2]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")          # write figures to file without a display
import matplotlib.pyplot as plt

import os
OUTPUT_DIR = "Code Outputs/Fusion Outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
def out(n): return os.path.join(OUTPUT_DIR, n)

INPUT_FILE   = "Code Outputs/Lake Level Outputs/African_Great_Lakes_Water_Levels.xlsx"   #Collated file
BACKBONE_COL = "Copernicus_Level_m"           

# Sources used to fill Copernicus gaps
INFILL_ORDER = ["GREALM_Level_m", "DAHITI_Level_m"]

NICE = {"Copernicus_Level_m": "Copernicus",
        "GREALM_Level_m": "GREALM",
        "DAHITI_Level_m": "DAHITI"}
MIN_OVERLAP = 6   # Use at least this many overlapping months to trust an offset

# Loading master file and aggregating to monthly  means

df = pd.read_excel(INPUT_FILE)
df["Date"]  = pd.to_datetime(df["Date"])
df["Month"] = df["Date"].dt.to_period("M").dt.to_timestamp()  # 1st of each month

source_cols = list(NICE.keys())
monthly = (df.groupby(["Reservoir", "Month"])[source_cols]
             .mean()
             .reset_index())

# Lake by lake bias align the sources to the bacbkbone then fill in the gaps
offset_rows    = []   # consistency/offset diagnostics
unified_frames = []   # the fused output per lake

for lake, g in monthly.groupby("Reservoir"):
    # Put the lake on a continuous monthly index so gaps show as NaN
    g = g.sort_values("Month").set_index("Month")
    full_index = pd.date_range(g.index.min(), g.index.max(), freq="MS")
    g = g.reindex(full_index)

    backbone = g[BACKBONE_COL].copy()

    # Align each source to the backbone
    aligned = {}
    for src in INFILL_ORDER:
        both = g[[BACKBONE_COL, src]].dropna()          # months where both exist
        if len(both) >= MIN_OVERLAP:
            diff   = both[src] - both[BACKBONE_COL]      # source-minus-backbone
            offset = diff.median()                       # offset
            corr   = both[BACKBONE_COL].corr(both[src])  # shape agreement
            aligned[src] = g[src] - offset               # shifted
            offset_rows.append({
                "Lake": lake, "Source": NICE[src],
                "Overlap_months": len(both),
                "Median_offset_m": round(offset, 3),
                "Offset_std_m": round(diff.std(), 3),    # residual scatter
                "Overlap_corr": round(corr, 3),
            })
        else:
            # not enough overlap to align safely so do not use this source here
            aligned[src] = pd.Series(np.nan, index=g.index)
            offset_rows.append({
                "Lake": lake, "Source": NICE[src],
                "Overlap_months": len(both),
                "Median_offset_m": np.nan, "Offset_std_m": np.nan,
                "Overlap_corr": np.nan,
            })

    # Build the unified series, backbone first, then aligned infill
    unified     = backbone.copy()
    source_flag = pd.Series(index=g.index, dtype=object)
    source_flag[backbone.notna()] = "Copernicus"

    for src in INFILL_ORDER:
        fill_mask = unified.isna() & aligned[src].notna()   # only fill real gaps
        unified[fill_mask]     = aligned[src][fill_mask]
        source_flag[fill_mask] = NICE[src] + "_aligned"

    unified_frames.append(pd.DataFrame({
        "Date": g.index, "Reservoir": lake,
        "Copernicus_m":     g[BACKBONE_COL].values,
        "GREALM_raw_m":     g["GREALM_Level_m"].values,
        "DAHITI_raw_m":     g["DAHITI_Level_m"].values,
        "GREALM_aligned_m": aligned["GREALM_Level_m"].values,
        "DAHITI_aligned_m": aligned["DAHITI_Level_m"].values,
        "Unified_Level_m":  unified.values,
        "Source":           source_flag.values,
        "Copernicus_only_m": g[BACKBONE_COL].values,   # for sensitivity analysis
    }))

uni     = pd.concat(unified_frames, ignore_index=True)
offsets = pd.DataFrame(offset_rows)

uni.to_excel(out("Unified_BiasAligned_Levels.xlsx"), index=False)
offsets.to_csv(out("Fusion_Offsets_Consistency.csv"), index=False)

# Reprots on how many motnhs each source covered
print("=== UNIFIED COVERAGE & SOURCE COMPOSITION ===")
for lake, g in uni.groupby("Reservoir"):
    tot      = g["Unified_Level_m"].notna().sum()
    cop_only = g["Copernicus_only_m"].notna().sum()
    comp     = g["Source"].value_counts().to_dict()
    print(f"{lake:16s} unified={tot:3d}  copernicus_only={cop_only:3d}  "
          f"+{tot - cop_only:3d} infilled  {comp}")

print("\n=== BIAS OFFSETS & OVERLAP CONSISTENCY ===")
print(offsets.to_string(index=False))

# test for homogeneoty compare month-to-month step at source changes vs typical within-source
# ---------------------------------------------------------------------------
lakes = list(uni["Reservoir"].unique())
rows  = []
for lake in lakes:
    g    = uni[uni["Reservoir"] == lake].sort_values("Date").reset_index(drop=True)
    lv   = g["Unified_Level_m"].values
    src  = g["Source"].values
    steps      = np.abs(np.diff(lv))            # change between consecutive months
    src_change = src[1:] != src[:-1]            # True where the source switched
    typical    = np.nanmedian(steps[~src_change])
    trans      = steps[src_change]
    rows.append({
        "Lake": lake,
        "typical_step_m": round(typical, 3),
        "n_transitions": int(src_change.sum()),
        "median_transition_step_m": round(np.nanmedian(trans), 3) if len(trans) else np.nan,
        "max_transition_step_m":    round(np.nanmax(trans), 3)    if len(trans) else np.nan,
    })
chk = pd.DataFrame(rows)
chk.to_csv(out("Fusion_TransitionJump_Check.csv"), index=False)
print("\n=== TRANSITION-JUMP CHECK ===")
print(chk.to_string(index=False))

# Figure 1, sources overlaid with the unified series and infilled points
fig, axes = plt.subplots(4, 2, figsize=(15, 16)); axes = axes.flatten()
for i, lake in enumerate(lakes):
    g = uni[uni["Reservoir"] == lake].sort_values("Date"); ax = axes[i]
    ax.plot(g["Date"], g["GREALM_aligned_m"], ".", ms=3, color="tab:green",
            alpha=.5, label="G-REALM (aligned)")
    ax.plot(g["Date"], g["DAHITI_aligned_m"], ".", ms=3, color="tab:orange",
            alpha=.5, label="DAHITI (aligned)")
    ax.plot(g["Date"], g["Copernicus_m"], "-", lw=.8, color="tab:blue",
            alpha=.7, label="Copernicus (backbone)")
    infill = g[g["Source"] != "Copernicus"]
    ax.plot(infill["Date"], infill["Unified_Level_m"], "o", ms=4,
            color="red", label="infilled months")
    ax.set_title(lake, fontsize=11, fontweight="bold"); ax.set_ylabel("Level (m)")
    if i == 0:
        ax.legend(fontsize=7, loc="best")
axes[7].set_visible(False)
plt.suptitle("Bias-Aligned Source Consistency & Backbone Infill "
             "(Copernicus universal backbone)",
             fontsize=14, fontweight="bold", y=.995)
plt.tight_layout(); plt.savefig(out("Fusion_Diag_1_SourceOverlay.png"), dpi=800); plt.close()

# Figure 2, residual difference after alignment

fig, axes = plt.subplots(4, 2, figsize=(15, 14)); axes = axes.flatten()
for i, lake in enumerate(lakes):
    g = uni[uni["Reservoir"] == lake].sort_values("Date"); ax = axes[i]
    for src, c in [("GREALM_aligned_m", "tab:green"), ("DAHITI_aligned_m", "tab:orange")]:
        ax.plot(g["Date"], g[src] - g["Copernicus_m"], ".", ms=3, color=c,
                alpha=.6, label=src.split("_")[0])
    ax.axhline(0, color="k", lw=.8); ax.set_ylim(-1, 1)
    ax.set_title(lake, fontsize=11, fontweight="bold")
    ax.set_ylabel("aligned - backbone (m)")
    if i == 0:
        ax.legend(fontsize=8)
axes[7].set_visible(False)
plt.suptitle("Residual Difference After Bias Alignment "
             "(should hover around 0, no drift)",
             fontsize=14, fontweight="bold", y=.995)
plt.tight_layout(); plt.savefig(out("Fusion_Diag_2_ResidualDiff.png"), dpi=800); plt.close()

=== UNIFIED COVERAGE & SOURCE COMPOSITION ===
Lake Albert      unified=355  copernicus_only=328  + 27 infilled  {'Copernicus': 328, 'GREALM_aligned': 15, 'DAHITI_aligned': 12}
Lake Edward      unified=330  copernicus_only=306  + 24 infilled  {'Copernicus': 306, 'GREALM_aligned': 15, 'DAHITI_aligned': 9}
Lake Kivu        unified=317  copernicus_only=270  + 47 infilled  {'Copernicus': 270, 'DAHITI_aligned': 29, 'GREALM_aligned': 18}
Lake Malawi      unified=403  copernicus_only=385  + 18 infilled  {'Copernicus': 385, 'GREALM_aligned': 18}
Lake Tanganyika  unified=401  copernicus_only=386  + 15 infilled  {'Copernicus': 386, 'GREALM_aligned': 15}
Lake Turkana     unified=401  copernicus_only=385  + 16 infilled  {'Copernicus': 385, 'GREALM_aligned': 16}
Lake Victoria    unified=402  copernicus_only=387  + 15 infilled  {'Copernicus': 387, 'GREALM_aligned': 15}

=== BIAS OFFSETS & OVERLAP CONSISTENCY ===
           Lake Source  Overlap_months  Median_offset_m  Offset_std_m  Overlap_corr
    L